# ML Modelling: Picking the Right Model *and* the Right Metric

**Dataset:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), loaded via `sklearn.datasets.load_breast_cancer` (569 real patient samples, 30 numeric features computed from digitized fine-needle-aspirate images, binary label: malignant vs. benign).

## Problem statement

A hospital wants a model that flags a tumor as **malignant** from the digitized cell-nuclei measurements. Three candidate models have been proposed by different team members, each reporting a single "accuracy" number and asking you — the QA-turned-ML-quality-engineer — to sign off on which one should ship.

Your SQA instinct should immediately object: *accuracy alone is not an acceptable acceptance criterion for a medical screening model.* A model that misses a malignant tumor (false negative) is far more costly than one that raises a false alarm (false positive), and a single aggregate number can hide instability across different patient subsamples.

## What you should produce

1. A reusable `evaluate_model(...)` function that scores a fitted classifier on **multiple** metrics (accuracy, precision, recall, F1, ROC-AUC, PR-AUC) plus a confusion matrix — not just accuracy.
2. A reusable `cross_val_stability(...)` function that runs stratified k-fold cross-validation and reports the **mean and standard deviation** of ROC-AUC across folds, flagging any model as `"UNSTABLE"` if its fold-to-fold std exceeds a threshold — this is the ML analogue of a flaky test in SQA.
3. A **model comparison report** (pandas DataFrame) across Logistic Regression, Random Forest, and Gradient Boosting, trained with a fixed `random_state` for reproducibility.
4. A **cost-sensitive threshold search**: sweeping the decision threshold of the best model and picking the one that minimizes total cost under the assumption that a false negative costs 10x a false positive (a realistic medical-screening cost structure), rather than blindly using the default 0.5 cutoff.
5. A one-paragraph written recommendation: which model should ship, at which threshold, and why — citing the specific numbers from your report.

By the end you should be able to explain, in plain language, why "99% accurate" is not sufficient evidence to approve a model for release.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

RANDOM_STATE = 42  # fixed for reproducibility -- never leave this unset in a real eval harness


### Load and split the data

Real patient data, real feature names. Keep a held-out test set untouched until final evaluation — no peeking, no re-fitting on it, no "just one more look".

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target  # y: 0 = malignant, 1 = benign (per sklearn's encoding)

print(X.shape, y.value_counts().to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)


### TODO 1 — `evaluate_model`

Fill in the function below. It should:
- fit nothing (assume `model` is already fitted),
- predict labels AND probabilities on `X_test`,
- return a `dict` with keys: `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`, `confusion_matrix`.

Think about *why* precision/recall/ROC-AUC/PR-AUC can disagree with each other and with accuracy — you'll need that intuition for the write-up.

In [ ]:
def evaluate_model(model, X_test, y_test):
    \"\"\"Score a fitted binary classifier on multiple metrics.

    Returns a dict with keys:
        accuracy, precision, recall, f1, roc_auc, pr_auc, confusion_matrix
    \"\"\"
    # TODO: predict labels with model.predict(X_test)
    # TODO: predict probabilities with model.predict_proba(X_test) -- use the
    #       column for the POSITIVE class consistently (decide which class is
    #       "positive" for this problem and say so in a comment)
    # TODO: compute each metric and return them in a dict
    raise NotImplementedError


### TODO 2 — `cross_val_stability`

Fill in the function below. It should:
- run `StratifiedKFold` cross-validation with `n_splits=cv` on the **training** data,
- score each fold on ROC-AUC,
- return `(mean_auc, std_auc, is_stable)` where `is_stable` is `False` if `std_auc > std_threshold`.

This mirrors flaky-test detection: a model whose score swings wildly between folds is not ready to ship, even if its average looks fine.

In [ ]:
def cross_val_stability(model, X, y, cv=5, std_threshold=0.03, random_state=RANDOM_STATE):
    \"\"\"Return (mean_auc, std_auc, is_stable) from stratified k-fold CV on ROC-AUC.\"\"\"
    # TODO: build a StratifiedKFold with n_splits=cv, shuffle=True, random_state=random_state
    # TODO: use cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    # TODO: compute mean and std of the fold scores
    # TODO: is_stable = std <= std_threshold
    raise NotImplementedError


### TODO 3 — Build and compare the three candidate models

Train `LogisticRegression` (wrapped with `StandardScaler` in a pipeline — it's scale-sensitive), `RandomForestClassifier`, and `GradientBoostingClassifier`, each with `random_state=RANDOM_STATE`. Run both of your functions on each and assemble a comparison `pandas.DataFrame` with one row per model.

In [ ]:
models = {
    # TODO: "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(...)),
    # TODO: "random_forest": RandomForestClassifier(...),
    # TODO: "gradient_boosting": GradientBoostingClassifier(...),
}

results = []
# TODO: for each model: fit on X_train/y_train, call evaluate_model and
#       cross_val_stability, and append a row (as a dict) to `results`

comparison_df = pd.DataFrame(results)
comparison_df


### TODO 4 — Cost-sensitive threshold search

Pick the model your report above suggests is best. Sweep classification thresholds from 0.01 to 0.99 and, for each, compute the confusion matrix on the test set and a **total cost** of `10 * false_negatives + 1 * false_positives` (malignant missed is 10x worse than a false alarm — tune this ratio if you disagree, but justify it). Report the threshold that minimizes total cost, and compare it against the default 0.5 threshold.

In [ ]:
def threshold_cost_search(model, X_test, y_test, fn_cost=10, fp_cost=1, thresholds=None):
    \"\"\"Return a DataFrame of (threshold, fn, fp, total_cost) and the best threshold.\"\"\"
    # TODO: get predicted probabilities for the positive class
    # TODO: for each threshold in `thresholds`, derive predictions and the
    #       confusion matrix, compute total_cost, and collect a row
    # TODO: find and return the threshold with minimum total_cost
    raise NotImplementedError


### TODO 5 — Write your recommendation

In a markdown cell, state: which model ships, at which decision threshold, and why — reference the specific numbers from `comparison_df` and your cost search. Call out any model you rejected for being unstable (high CV std) even if its point-estimate accuracy looked attractive.

*(Your recommendation here before checking the solution.)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**`evaluate_model`, `cross_val_stability`, model comparison, and threshold search:**

```python
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # probability of class 1 ("benign")
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

def cross_val_stability(model, X, y, cv=5, std_threshold=0.03, random_state=42):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(model, X, y, cv=skf, scoring="roc_auc")
    mean_auc, std_auc = scores.mean(), scores.std()
    return mean_auc, std_auc, std_auc <= std_threshold

models = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(random_state=42, max_iter=1000)),
    "random_forest": RandomForestClassifier(random_state=42, n_estimators=200),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    mean_auc, std_auc, is_stable = cross_val_stability(model, X_train, y_train)
    results.append({
        "model": name,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "cv_auc_mean": mean_auc,
        "cv_auc_std": std_auc,
        "stable": is_stable,
    })

comparison_df = pd.DataFrame(results)

def threshold_cost_search(model, X_test, y_test, fn_cost=10, fp_cost=1, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    y_proba = model.predict_proba(X_test)[:, 1]
    rows = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        total_cost = fn_cost * fn + fp_cost * fp
        rows.append({"threshold": t, "fn": fn, "fp": fp, "total_cost": total_cost})
    df = pd.DataFrame(rows)
    best_row = df.loc[df["total_cost"].idxmin()]
    return df, best_row
```

**Why this matters for ML quality engineering:**

- **Accuracy alone hides risk.** On this dataset accuracy, precision and recall for the strongest model are all close (breast cancer data is fairly separable), but recall on the malignant class is the number that actually matters clinically — a confusion matrix makes the false-negative count visible where a single accuracy score does not.
- **`cv_auc_std` is your flakiness signal.** A model with a high mean ROC-AUC but a large standard deviation across folds is analogous to a flaky test: it "passes" on average but you can't trust any single run. Reject it even if its point estimate looks best.
- **The default 0.5 threshold is an arbitrary engineering default, not a business requirement.** Once you attach real costs to false negatives vs. false positives, the optimal threshold usually moves — here, lower, to catch more malignant cases at the price of a few more unnecessary biopsies, which is the right trade-off for a screening tool.
- **Reproducibility.** Every model and every CV split uses the same fixed `random_state`. Without it, "the model got 97% accuracy" is not a result you can audit or reproduce — a core QA requirement carried straight over from software testing.

</details>